# LiveSight Phase 2: Multi-Object Tracking with ByteTrack (GPU Experiments)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

This Google Colab notebook enables GPU-accelerated experiments for **LiveSight Phase 2: Multi-Object Tracking** using pretrained **Ultralytics YOLO** and **ByteTrack** (`bytetrack.yaml`).

### Objectives:
- Run YOLO object detection and ByteTrack on an NVIDIA GPU (e.g. T4 / V100 / A100).
- Annotate user-supplied videos (such as `video/gettyimages-1206324549-640_adpp.mp4`).
- Measure GPU throughput (FPS) and tracking latency.
- Observe track IDs, occlusion re-acquisition, and tracker reset behavior.
- Visualize annotated tracking videos directly in Colab.

## 1. Verify GPU Hardware Acceleration
Ensure your Colab runtime is set to **GPU**: *Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU*.

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
else:
    print("Running on CPU mode. For GPU acceleration, switch runtime to GPU in Colab (Runtime -> Change runtime type).")

## 2. Install Dependencies
Install `ultralytics`, `opencv-python`, and `lapx` (the fast C++ linear assignment solver required for ByteTrack).

In [ ]:
!pip install --upgrade pip
!pip install ultralytics opencv-python lapx

## 3. Clone Repository or Setup Project Files
If your repository is hosted on GitHub, you can clone it:
```bash
!git clone https://github.com/your-username/LiveSight.git
%cd LiveSight
```
The cell below ensures the directories and default ByteTrack configuration are present.

In [ ]:
import os
from pathlib import Path

# Create project directory structure
Path("data").mkdir(exist_ok=True)
Path("video").mkdir(exist_ok=True)
Path("configs").mkdir(exist_ok=True)

# Create default ByteTrack configuration
bytetrack_yaml = """
tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: 30
match_thresh: 0.8
fuse_score: True
"""
Path("configs/bytetrack.yaml").write_text(bytetrack_yaml.strip())
print("Config ready: configs/bytetrack.yaml")

## 4. Video Selection
Select the input video. If `video/gettyimages-1206324549-640_adpp.mp4` exists, it will be used automatically. Otherwise, you can upload your own or generate a synthetic test clip.

In [ ]:
import glob
from pathlib import Path

# 1. Check for existing video in video/ folder
video_candidates = list(Path("video").glob("*.mp4"))
input_video_path = None

for cand in video_candidates:
    if "annotated" not in cand.name:
        input_video_path = str(cand)
        break

if input_video_path and Path(input_video_path).exists():
    print(f"[+] Found input video: {input_video_path}")
else:
    print("[*] No input video found in video/. Creating synthetic sample video...")
    import cv2
    import numpy as np
    input_video_path = "video/sample_demo.mp4"
    out = cv2.VideoWriter(input_video_path, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (640, 480))
    for i in range(120):
        frame = np.zeros((480, 640, 3), dtype=np.uint8)
        x1, y1 = int(50 + (i * 4) % 450), int(80 + (i * 2) % 300)
        cv2.rectangle(frame, (x1, y1), (x1 + 80, y1 + 100), (0, 220, 255), -1)
        out.write(frame)
    out.release()
    print(f"[+] Created synthetic video: {input_video_path}")

## 5. Run GPU-Accelerated ByteTrack Tracking & Annotation
Instantiate YOLO with `device='cuda:0'` (falling back to `'cpu'` if CUDA is not active) and run ByteTrack multi-object tracking to annotate the video.

In [ ]:
import time
import cv2
from ultralytics import YOLO

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Inference Device: {device}")

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture(input_video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_path = "video/annotated_output_colab.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print(f"Processing: {input_video_path} ({width}x{height} @ {fps:.1f} FPS, {total_frames} frames)...")

frame_count = 0
start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run tracking with ByteTrack
    results = model.track(
        frame,
        persist=True,
        tracker="configs/bytetrack.yaml",
        conf=0.35,
        device=device,
        verbose=False,
    )
    
    # Draw annotated boxes and track IDs
    annotated_frame = results[0].plot()
    writer.write(annotated_frame)
    frame_count += 1
    
    if frame_count % 50 == 0 or frame_count == total_frames:
        elapsed_so_far = time.time() - start_time
        current_fps = frame_count / elapsed_so_far if elapsed_so_far > 0 else 0
        print(f"  Progress: {frame_count}/{total_frames} frames ({frame_count/total_frames*100:.1f}%) | {current_fps:.1f} FPS")

cap.release()
writer.release()

elapsed = time.time() - start_time
avg_fps = frame_count / elapsed if elapsed > 0 else 0

print(f"\n=== Annotation & Tracking Run Summary ===")
print(f"Device:            {device}")
print(f"Frames Processed:  {frame_count}")
print(f"Total Time:        {elapsed:.2f} s")
print(f"Average FPS:       {avg_fps:.1f}")
print(f"Output Saved To:   {out_path}")

## 6. Video Playback in Notebook
Convert the output to H.264 MP4 to view it directly in Google Colab.

In [ ]:
# Convert to browser-compatible H.264 video
!ffmpeg -y -i video/annotated_output_colab.mp4 -vcodec libx264 video/display_video.mp4

from IPython.display import HTML
from base64 import b64encode

if Path("video/display_video.mp4").exists():
    mp4 = open('video/display_video.mp4','rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=640 controls autoplay loop>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("video/display_video.mp4 not found. Please verify video annotation step completed.")

## 7. Observations & Tracker Limitations
- **Track Scoping**: Track IDs are local to the continuous video stream session and reset when the stream restarts.
- **Occlusion Limit**: If an object disappears for longer than `track_buffer` (30 frames), a new track ID is assigned when it reappears.
- **Crossing Paths**: Fast motion or dense overlap can occasionally cause ID switches; tuning `match_thresh` and `track_high_thresh` in `bytetrack.yaml` can reduce drift.